# Sampling

The purpose of this notebook is to draw a balanced sample from the bulk data in `../data/bulk/`: `SAMPLE_PER_MODE` replays per `fastBuilderMode`. The sample is written to `../data/sampled_replays.jsonl` (one replay per line), which `02_preprocessing` reads line by line.

The bulk data (~26 GB) doesn't fit into memory, so every step goes through the files one at a time.

**Index**

- libraries
- settings
- 01 - INDEX ALL REPLAYS
- 02 - SAMPLING
- 03 - WRITE SAMPLE
- 04 - SANITY CHECKS


## libraries

In [ ]:
import json
import pandas as pd

from pathlib import Path

## settings

In [ ]:
DATA_DIR = Path("../data")
BULK_DIR = DATA_DIR / "bulk"
SAMPLE_PATH = DATA_DIR / "sampled_replays.jsonl"

SAMPLE_PER_MODE = 10000
RANDOM_SEED = 18

## 01 - index all replays
first pass: collect `idAnonymous`, `fastBuilderMode` and the file name of every replay

In [ ]:
data_file_paths = sorted(BULK_DIR.glob("*.json"))

rows = []
for path in data_file_paths:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        rows.append({
            "idAnonymous": item["idAnonymous"],
            "fastBuilderMode": item["fastBuilderMode"],
            "file": path.name,
        })
    del data
    print(f"{path.name}: {len(rows)} replays indexed so far")

df_index=pd.DataFrame(rows)
df_index["fastBuilderMode"].value_counts()


## 02 - sampling
draw `SAMPLE_PER_MODE` replays per mode

In [ ]:
df_sample = (df_index.groupby("fastBuilderMode")
             .sample(n=SAMPLE_PER_MODE, random_state=RANDOM_SEED))
sampled_ids = set(df_sample["idAnonymous"])

print(df_sample["fastBuilderMode"].value_counts())

## 03 - write sample
second pass: keep only the sampled replays and write them to the `.jsonl` file

#### writing the file

In [ ]:
with open(SAMPLE_PATH, "w", encoding="utf-8") as out:
    for path in data_file_paths:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)

        for item in data:
            if item["idAnonymous"] in sampled_ids:
                out.write(json.dumps(item) + "\n")

        del data
        print(f"{path.name} done")

## 04 - sanity checks
read the `.jsonl` back and check the replay count per mode

In [ ]:
line_modes = []
with open(SAMPLE_PATH, encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        line_modes.append(item["fastBuilderMode"])

series = pd.Series(line_modes)
series.value_counts()
